# LinCIR trên Google Colab Pro

Notebook này dùng **mã công bố chính thức** của LinCIR với CLIP ViT-L/14 để:

1. tải checkpoint `lincir_large.pt` và chạy smoke test thật;
2. đánh giá trên CIRR validation khi bạn đã đặt dataset trong Google Drive;
3. tùy chọn huấn luyện lại 20.000 bước và lưu checkpoint vào Drive.

Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Chạy lần lượt từ trên xuống. Mã nguồn trong repo LinCIR mang giấy phép **CC BY-NC 4.0**; hãy kiểm tra riêng model card trước khi dùng checkpoint ngoài nghiên cứu.

## 0. Gắn Drive và cấu hình

Để dùng Drive, giữ `USE_DRIVE = True`. Nếu Colab báo `credential propagation was unsuccessful`, cell sẽ tự dùng `/content/LinCIR` để bạn vẫn chạy được smoke test; dữ liệu trong `/content` sẽ mất khi runtime bị xóa. Sửa `CIRR_ROOT` nếu dataset của bạn nằm ở vị trí khác.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import os
import shutil
import subprocess
import sys
os.environ['USE_TF'] = '0'  # CLIP trong notebook chỉ dùng PyTorch

import torch

USE_DRIVE = True
DRIVE_MOUNTED = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
    except Exception as exc:
        print(f'Không gắn được Drive ({exc}). Tiếp tục bằng bộ nhớ tạm /content.')

if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.')

WORK = Path('/content/lincir')
DRIVE_ROOT = Path('/content/drive/MyDrive/LinCIR') if DRIVE_MOUNTED else Path('/content/LinCIR')
CIRR_ROOT = (Path('/content/drive/MyDrive') if DRIVE_MOUNTED else Path('/content')) / 'datasets/CIRR'  # sửa nếu cần
CACHE_DIR = DRIVE_ROOT / 'hf_cache'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RUN_DIR = DRIVE_ROOT / 'runs' / 'vit_l_official'
for directory in (CACHE_DIR, CHECKPOINT_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
TRAIN_BATCH_SIZE = 128 if vram_gb >= 35 else 64 if vram_gb >= 22 else 32
EFFECTIVE_BATCH_SIZE = 512
assert EFFECTIVE_BATCH_SIZE % TRAIN_BATCH_SIZE == 0
ACCUMULATION = EFFECTIVE_BATCH_SIZE // TRAIN_BATCH_SIZE
TRAIN_DIR = RUN_DIR / 'training_seed12345'  # đổi tên cho mỗi lần huấn luyện mới
print(f'GPU: {gpu_name} ({vram_gb:.1f} GiB); batch huấn luyện đề xuất: {TRAIN_BATCH_SIZE}')


## 1. Lấy đúng phiên bản mã và cài phụ thuộc

Khóa commit LinCIR và phiên bản thư viện. Cell vá sửa tương thích CLIPEncoder, tích lũy gradient, tên biến, hai nguồn caption trong bài báo, và dừng khi ảnh lỗi. Giữ kiến trúc Phi/loss SMP của mã công bố.


In [ ]:
REPO = 'https://github.com/navervision/lincir.git'
UPSTREAM_COMMIT = '1dec42d118da816be2a43fd43cc0746e05f63881'
if not WORK.exists():
    subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', UPSTREAM_COMMIT], check=True)

packages = [
    'transformers==4.57.6', 'datasets==4.8.4', 'accelerate==1.12.0',
    'spacy==3.8.16', 'huggingface_hub==0.36.2', 'webdataset>=0.2,<1',
    'ftfy>=6,<7', 'tensorboard>=2.16',
    'git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

import transformers, accelerate
if transformers.__version__ != '4.57.6' or accelerate.__version__ != '1.12.0':
    raise RuntimeError('Thư viện cũ còn trong RAM: restart session rồi chạy lại từ mục 0.')

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
commit = subprocess.run(
    ['git', '-C', str(WORK), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
assert commit == UPSTREAM_COMMIT
print('Mã nguồn sẵn sàng:', commit)


In [ ]:
# Compatibility patches for the pinned upstream checkout; safe to rerun.
def patch_upstream(work):
    def replace(filename, before, after):
        path = work / filename
        source = path.read_text(encoding='utf-8')
        if after in source:
            return
        if before in source:
            if source.count(before) != 1:
                raise RuntimeError(f'Unexpected upstream source: {filename}')
            path.write_text(source.replace(before, after), encoding='utf-8')
        elif after not in source:
            raise RuntimeError(f'Patch no longer applies: {filename}')

    replace('encode_with_pseudo_tokens.py',
            'output_hidden_states=False,\n                                      return_dict=False)',
            'output_hidden_states=False)')
    replace('train_phi.py', 'phi.parameters(), arg.max_grad_norm)', 'phi.parameters(), args.max_grad_norm)')
    replace('train_phi.py', 'from accelerate.utils import set_seed',
            'from accelerate.utils import set_seed, GradientAccumulationPlugin')
    replace('train_phi.py',
            'gradient_accumulation_steps=args.gradient_accumulation_steps,',
            'gradient_accumulation_plugin=GradientAccumulationPlugin(\n'
            '                num_steps=args.gradient_accumulation_steps, sync_with_dataloader=False),')
    path = work / 'train_phi.py'
    source = path.read_text(encoding='utf-8')
    if 'with accelerator.accumulate(phi):' not in source:
        start = source.index('            original_tokens = original_tokens.to(')
        end = source.index('            if accelerator.sync_gradients:\n', start)
        body = source[start:end]
        source = source[:start] + '            with accelerator.accumulate(phi):\n' + ''.join(
            '    ' + line if line.strip() else line for line in body.splitlines(keepends=True)
        ) + source[end:]
        path.write_text(source, encoding='utf-8')
    replace('train_phi.py', '            if accelerator.sync_gradients:\n',
            '            if accelerator.optimizer_step_was_skipped:\n'
            "                raise FloatingPointError('fp16 overflow: retry with bf16/no precision.')\n"
            '            if accelerator.sync_gradients:\n')
    replace('train_phi.py', '                exit()',
            '                accelerator.end_training()\n                return')
    replace('loader.py',
            "                  'dataset3': 'Geonmo/midjourney-prompts-only',\n",
            '                  # Paper sources: CC3M + Stable Diffusion Prompts only.\n')
    replace('data_utils.py', '    batch = list(filter(lambda x: x is not None, batch))',
            "    if any(item is None for item in batch):\n"
            "        raise ValueError('Image decoding failed; refusing partial benchmark results.')")

patch_upstream(WORK)
print('Đã áp dụng các vá tương thích.')


## 2. Tải model ViT-L do tác giả công bố

File được chép vào Drive, nên không phải tải lại sau khi Colab ngắt phiên.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REVISION = '5118df4683de6efa09f8e5336d86ca626ed21c44'
EXPECTED_SHA256 = '5ba98d52db7a5e9a78f9ba7436991235477b89544caf246e4659642bf9b409bd'
downloaded = Path(hf_hub_download(
    repo_id='navervision/zeroshot-cir-models',
    filename='lincir_large.pt',
    revision=MODEL_REVISION,
    cache_dir=str(CACHE_DIR),
))
CHECKPOINT = CHECKPOINT_DIR / 'lincir_large.pt'
if not CHECKPOINT.exists():
    shutil.copy2(downloaded, CHECKPOINT)
if CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError('Checkpoint tải về có kích thước bất thường.')
with CHECKPOINT.open('rb') as stream:
    checkpoint_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
if checkpoint_sha256 != EXPECTED_SHA256:
    raise RuntimeError(f'SHA256 checkpoint không khớp: {checkpoint_sha256}')
print(CHECKPOINT, f'{CHECKPOINT.stat().st_size / 2**20:.1f} MiB', checkpoint_sha256)


## 3. Smoke test checkpoint

Cell này chạy ảnh mẫu qua CLIP image encoder → Phi → CLIP text encoder. Nó kiểm tra model tải đúng và đầu ra hữu hạn; đây chưa phải đánh giá chất lượng.

In [ ]:
import clip
from PIL import Image
from encode_with_pseudo_tokens import encode_with_pseudo_tokens_HF
from models import Phi, build_text_encoder

device = torch.device('cuda')
args = SimpleNamespace(clip_model_name='large', mixed_precision='fp16', cache_dir=str(CACHE_DIR))
image_encoder, preprocess, text_encoder, _ = build_text_encoder(args)
phi = Phi(
    input_dim=text_encoder.config.projection_dim,
    hidden_dim=text_encoder.config.projection_dim * 4,
    output_dim=text_encoder.config.hidden_size,
    dropout=0.5,
)
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
phi.load_state_dict(state['Phi'])
image_encoder = image_encoder.to(device=device, dtype=torch.float16).eval()
text_encoder = text_encoder.to(device=device, dtype=torch.float16).eval()
phi = phi.to(device=device, dtype=torch.float16).eval()

with Image.open(WORK / 'example1.jpg') as image:
    pixels = preprocess(images=image.convert('RGB'), return_tensors='pt').pixel_values
pixels = pixels.to(device=device, dtype=torch.float16)
tokens = clip.tokenize(['a photo of $ that is a pencil sketch'], context_length=77).to(device)
with torch.inference_mode():
    image_features = image_encoder(pixel_values=pixels).image_embeds
    pseudo_token = phi(image_features)
    composed = encode_with_pseudo_tokens_HF(text_encoder, tokens, pseudo_token)
assert composed.shape == image_features.shape == (1, text_encoder.config.projection_dim)
assert torch.isfinite(composed).all()
print('PASS:', 'image', tuple(image_features.shape), 'pseudo-token', tuple(pseudo_token.shape),
      'composed', tuple(composed.shape))
del image_encoder, text_encoder, phi, state, pixels, tokens, image_features, pseudo_token, composed
torch.cuda.empty_cache()


## 4. Chuẩn bị CIRR

**Nếu chưa có dữ liệu:** cell bên dưới tự tải nhãn CIRR công khai, khóa phiên bản và kiểm tra SHA256. Ảnh gốc cần lấy riêng:

1. Điền [form xin dữ liệu NLVR2](https://docs.google.com/forms/d/e/1FAIpQLSdB_OhgmpQULV17kjQ4iitftILbOJjuGgJ2ECmg-HdmkjUSAg/viewform) theo [hướng dẫn của tác giả](https://github.com/lil-lab/nlvr/tree/master/nlvr2#direct-image-download), bằng thông tin của bạn và xác nhận điều khoản nghiên cứu.
2. Làm theo [hướng dẫn tải ảnh CIRR](https://cirr.zheyuanliu.me/raw-image-download). Nếu chưa nhận được phản hồi từ NLVR2, trang này hướng dẫn liên hệ CIRR để xin mật khẩu sau khi đã gửi form và đồng ý điều khoản.
3. Tải archive được cấp quyền, đối chiếu checksum do CIRR cung cấp, giải nén và đặt thư mục `dev` vào `MyDrive/datasets/CIRR/dev`. Giữ nguyên tên ảnh; chỉ cần dev cho bước validation đầu tiên. Không dùng tập ảnh tải thiếu từ URL cũ để báo kết quả baseline.

Có thể chạy mục 0–3 trong lúc chờ dữ liệu. CIRR validation cần **4.181 truy vấn và đủ 2.297 ảnh gallery**, theo nhãn chính thức. Cấu trúc tối thiểu:

```text
CIRR/
├── cirr/
│   ├── captions/cap.rc2.val.json
│   └── image_splits/split.rc2.val.json
└── dev/... ảnh được các JSON tham chiếu ...
```

Dữ liệu không được đóng gói trong notebook vì điều khoản phân phối của dataset.

In [ ]:
from urllib.request import urlopen

CIRR_REVISION = '2276fba2d2c53862c8ae79ea18e1f1ae03719f56'
annotation_hashes = {
    'captions/cap.rc2.val.json': 'a85c3a1aa464f1af7229918e8018d08b8b20ce5dab479ffdf39d61113140f919',
    'image_splits/split.rc2.val.json': 'de36f7a7219c8bb06347a67748cfc68405be7dad62853a1e31846c98e491235b',
    'captions/cap.rc2.test1.json': '26a2b2e7a5c96c903491aefc7d5f237f03d3cca97712e1b6bb3b231db5ff8001',
    'image_splits/split.rc2.test1.json': 'bf2bb9ed0659c4aae35de72350652b9ab75d36a060211f826bec15c58ec12566',
    'LICENSE': '7d96be4d31b933d681f4a30fff9e3551a850bfd9b0c31d649ee8a267ac058827',
}
for relative_path, expected_hash in annotation_hashes.items():
    destination = CIRR_ROOT / 'cirr' / relative_path
    if destination.is_file():
        content = destination.read_bytes()
    else:
        url = f'https://raw.githubusercontent.com/Cuberick-Orion/CIRR/{CIRR_REVISION}/{relative_path}'
        with urlopen(url, timeout=60) as response:
            content = response.read()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f'Nhãn CIRR không khớp phiên bản: {destination}')
    if not destination.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
print('Nhãn CIRR đã được xác minh:', CIRR_REVISION)


In [ ]:
from datetime import datetime, timezone
from PIL import Image

def check_cirr(split='val', expected_queries=4181, expected_gallery=2297):
    caption_file = CIRR_ROOT / f'cirr/captions/cap.rc2.{split}.json'
    split_file = CIRR_ROOT / f'cirr/image_splits/split.rc2.{split}.json'
    records, image_paths = [], {}
    missing, invalid = [], []
    if caption_file.is_file() and split_file.is_file():
        records = json.loads(caption_file.read_text(encoding='utf-8'))
        image_paths = json.loads(split_file.read_text(encoding='utf-8'))
        for relative_path in image_paths.values():
            path = CIRR_ROOT / relative_path
            if not path.resolve().is_relative_to(CIRR_ROOT.resolve()):
                raise ValueError(f'Đường dẫn ảnh nằm ngoài CIRR_ROOT: {relative_path}')
            if not path.is_file():
                missing.append(relative_path)
                continue
            try:
                with Image.open(path) as image:
                    image.convert('RGB').load()
            except (OSError, ValueError) as exc:
                invalid.append({'path': relative_path, 'error': str(exc)})
    CIRR_READY = len(records) == expected_queries and len(image_paths) == expected_gallery and not missing and not invalid
    readiness = {
        'status': 'READY' if CIRR_READY else 'NOT_READY',
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'root': str(CIRR_ROOT), 'queries': len(records), 'gallery_images': len(image_paths),
        'missing_images': missing, 'invalid_images': invalid,
    }
    (RUN_DIR / ('cirr_readiness.json' if split == 'val' else 'cirr_test1_readiness.json')).write_text(json.dumps(readiness, indent=2), encoding='utf-8')
    print(readiness['status'], f'{len(records)}/{expected_queries} truy vấn, {len(image_paths)}/{expected_gallery} ảnh trong nhãn; thiếu {len(missing)}, hỏng {len(invalid)} ảnh')
    if not CIRR_READY:
        print('Hoàn thành hướng dẫn lấy ảnh ở trên rồi chạy lại cell này. Ví dụ ảnh thiếu/hỏng:', missing[:3], invalid[:3])
    return CIRR_READY

CIRR_READY = check_cirr()


## 5. Đánh giá checkpoint công bố trên CIRR validation

Kết quả được lưu thành log trong Drive. Đây là **validation/dev**, không nên so trực tiếp với số test trong Bảng 6 của bài báo.

In [ ]:
def run_logged(command, log_path):
    from datetime import datetime, timezone
    from importlib.metadata import version
    log_path = Path(log_path)
    if log_path.exists():
        raise FileExistsError(f'Log đã tồn tại; chọn run mới: {log_path}')
    manifest = {
        'command': command, 'upstream_commit': UPSTREAM_COMMIT,
        'started_utc': datetime.now(timezone.utc).isoformat(),
        'gpu': gpu_name, 'effective_batch_size': EFFECTIVE_BATCH_SIZE,
        'packages': {name: version(name) for name in
                     ['torch', 'torchvision', 'transformers', 'accelerate', 'datasets', 'spacy']},
    }
    if '--phi-checkpoint-name' in command:
        checkpoint_path = Path(command[command.index('--phi-checkpoint-name') + 1])
        with checkpoint_path.open('rb') as stream:
            manifest['checkpoint_sha256'] = hashlib.file_digest(stream, 'sha256').hexdigest()
    manifest_path = log_path.with_suffix('.json')
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=WORK, text=True, encoding='utf-8',
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            returncode = process.wait()
    manifest.update(returncode=returncode, ended_utc=datetime.now(timezone.utc).isoformat())
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    if returncode:
        raise RuntimeError(f'Chạy thất bại (exit {returncode}), xem {log_path}.')

if not CIRR_READY:
    raise FileNotFoundError('CIRR chưa sẵn sàng; hoàn thành mục 4 trước.')
eval_command = [
    sys.executable, '-u', 'validate.py',
    '--eval-type', 'phi', '--dataset', 'cirr',
    '--dataset-path', str(CIRR_ROOT),
    '--phi-checkpoint-name', str(CHECKPOINT),
    '--clip_model_name', 'large', '--cache_dir', str(CACHE_DIR),
]
run_logged(eval_command, RUN_DIR / 'pretrained_cirr_val.log')


## 6. Huấn luyện lại (tùy chọn)

Chỉ bật sau khi smoke test và CIRR validation thành công. ViT-L, fp16, AdamW (LR 1e-4, weight decay 0.01), seed 12345, 20.000 optimizer steps, batch hiệu dụng 512 nhờ tích lũy gradient, kiểm tra/lưu mỗi 1.000 bước (mã gốc cũng kiểm tra ở bước 50). Chọn checkpoint có CIRR dev R@1 cao nhất.

Nếu thiếu VRAM, giảm TRAIN_BATCH_SIZE ở mục 0 xuống một ước của 512 rồi chạy lại. Tích lũy gradient không bảo đảm kết quả bit-exact với batch 512. Dùng toàn bộ caption CC3M + SDP; không cần ảnh train. Chọn High RAM nếu có. Colab Pro không bảo đảm một phiên đủ 20.000 bước.

Checkpoint gốc chỉ lưu trọng số, không đủ trạng thái resume; notebook không tự warm-start hoặc ghi đè run cũ. Đổi TRAIN_DIR cho lần chạy mới. Bài báo không nêu patience của early stopping; notebook theo ngân sách 20.000 bước trong README tác giả.


In [ ]:
RUN_FULL_TRAINING = False  # đổi thành True sau khi các mục trên đều PASS
MAX_TRAIN_STEPS = 20_000
if RUN_FULL_TRAINING:
    if not CIRR_READY:
        raise FileNotFoundError('CIRR chưa sẵn sàng; training của mã tác giả cần dev set để chọn model.')
    if TRAIN_DIR.exists() and any(TRAIN_DIR.iterdir()):
        raise FileExistsError(f'Giữ nguyên run cũ. Đổi TRAIN_DIR ở mục 0: {TRAIN_DIR}')
    train_command = [
        sys.executable, '-u', 'train_phi.py',
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--output_dir', str(TRAIN_DIR),
        '--cirr_dataset_path', str(CIRR_ROOT),
        '--cache_dir', str(CACHE_DIR),
        '--mixed_precision', 'fp16',
        '--clip_model_name', 'large',
        '--validation_steps', '1000',
        '--checkpointing_steps', '1000',
        '--learning_rate', '1e-4', '--weight_decay', '0.01',
        '--seed', '12345',
        '--num_workers', '2',
        '--lr_scheduler', 'constant_with_warmup',
        '--lr_warmup_steps', '0',
        '--gradient_accumulation_steps', str(ACCUMULATION),
        '--max_train_steps', str(MAX_TRAIN_STEPS),
    ]
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    run_logged(train_command, TRAIN_DIR / 'train.log')
else:
    print('Đã bỏ qua full training. Đổi RUN_FULL_TRAINING=True khi bạn sẵn sàng.')


## 7. Đánh giá model vừa huấn luyện

Dùng phi_best.pt được chọn bằng CIRR dev R@1; không tự thay bằng checkpoint cuối.


In [ ]:
EVALUATE_TRAINED = False  # đổi thành True sau khi training tạo checkpoint
if EVALUATE_TRAINED:
    if not CIRR_READY:
        raise FileNotFoundError('Hoàn thành kiểm tra dữ liệu ở mục 4.')
    trained_checkpoint = TRAIN_DIR / 'checkpoints/phi_best.pt'
    if not trained_checkpoint.is_file():
        raise FileNotFoundError('Chưa có checkpoint đã huấn luyện trong Drive.')
    trained_command = [
        sys.executable, '-u', 'validate.py',
        '--eval-type', 'phi', '--dataset', 'cirr',
        '--dataset-path', str(CIRR_ROOT),
        '--phi-checkpoint-name', str(trained_checkpoint),
        '--clip_model_name', 'large', '--cache_dir', str(CACHE_DIR),
    ]
    run_logged(trained_command, TRAIN_DIR / 'trained_cirr_val.log')
else:
    print('Đã bỏ qua đánh giá model tự train.')


## 8. Xuất dự đoán CIRR test (tùy chọn)

Để đối chiếu kết quả test của bài báo, cần thêm thư mục ảnh test1 lấy từ cùng nguồn CIRR, gồm 2.315 ảnh và 4.148 truy vấn. Mục 4 đã tải nhãn test1 có checksum. Đặt ảnh đúng đường dẫn trong split.rc2.test1.json.

Cell chỉ xuất hai file JSON vào Drive. Bạn nộp chúng qua cổng đánh giá CIRR liên kết từ [trang dataset](https://cirr.zheyuanliu.me/), giữ lại kết quả máy chủ và checkpoint tương ứng. Không có nhãn test để tính điểm cục bộ. Dùng checkpoint chính thức mặc định, hoặc đặt TEST_CHECKPOINT = TRAIN_DIR / 'checkpoints/phi_best.pt' sau huấn luyện. Không chọn siêu tham số bằng điểm test.


In [ ]:
EXPORT_CIRR_TEST = False
TEST_CHECKPOINT = CHECKPOINT
SUBMISSION_NAME = 'vit_l_official'  # đổi tên cho checkpoint/thí nghiệm khác
if EXPORT_CIRR_TEST:
    if not SUBMISSION_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in SUBMISSION_NAME):
        raise ValueError('SUBMISSION_NAME chỉ dùng chữ, số, dấu gạch ngang/gạch dưới.')
    if not check_cirr('test1', 4148, 2315):
        raise FileNotFoundError('CIRR test1 chưa đủ hoặc có ảnh hỏng.')
    if not Path(TEST_CHECKPOINT).is_file():
        raise FileNotFoundError(TEST_CHECKPOINT)
    submission_dir = RUN_DIR / 'submissions' / SUBMISSION_NAME
    if submission_dir.exists():
        raise FileExistsError(f'Giữ nguyên dự đoán cũ: {submission_dir}')
    submission_dir.mkdir(parents=True)
    command = [
        sys.executable, '-u', 'generate_test_submission.py',
        '--eval-type', 'phi', '--dataset', 'cirr',
        '--dataset-path', str(CIRR_ROOT), '--phi-checkpoint-name', str(TEST_CHECKPOINT),
        '--clip_model_name', 'large', '--cache_dir', str(CACHE_DIR),
        '--submission-name', SUBMISSION_NAME,
    ]
    run_logged(command, submission_dir / 'export.log')
    test_records = json.loads((CIRR_ROOT / 'cirr/captions/cap.rc2.test1.json').read_text())
    for prefix, metric, k in [('', 'recall', 50), ('subset_', 'recall_subset', 3)]:
        path = WORK / 'submission/cirr' / f'{prefix}{SUBMISSION_NAME}.json'
        predictions = json.loads(path.read_text())
        assert predictions['version'] == 'rc2' and predictions['metric'] == metric
        assert set(predictions) - {'version', 'metric'} == {str(r['pairid']) for r in test_records}
        for record in test_records:
            ranking = predictions[str(record['pairid'])]
            assert len(ranking) == len(set(ranking)) == k and record['reference'] not in ranking
            if prefix:
                assert set(ranking).issubset(record['img_set']['members'])
        shutil.copy2(path, submission_dir / path.name)
    print('Đã xuất dự đoán để bạn nộp CIRR:', submission_dir)
else:
    print('Đã bỏ qua xuất CIRR test.')


## Kết quả cần giữ lại

- Model công bố: `MyDrive/LinCIR/checkpoints/lincir_large.pt`
- Model tự train: `MyDrive/LinCIR/runs/vit_l_official/training_seed12345/checkpoints/phi_best.pt`
- Log baseline: `MyDrive/LinCIR/runs/vit_l_official/pretrained_cirr_val.log`
- Kiểm tra dữ liệu: `MyDrive/LinCIR/runs/vit_l_official/cirr_readiness.json`
- TensorBoard/checkpoint huấn luyện: bên trong thư mục run trên Drive

Khi baseline này chạy ổn định, hãy giữ nguyên seed/dataset/backbone/số bước và chỉ thay **một** thành phần cho mỗi thí nghiệm cải tiến.